In [1]:
import cv2
print("OpenCV should be 4.8.x Current version:", cv2.__version__)
from typing import List
import numpy as np
# import imageio --- IGNORE IF DEPRECATED---
import imageio.v3 as imageio
import copy
import os
import glob

OpenCV should be 4.8.x Current version: 4.8.0


In [11]:

# ============================================================
# 1. CONFIGURATION
# ============================================================

CAMERA_INDEX = 0

# Required security sequence
PASSWORD = [
    "RED_CROSS",
    "YELLOW_CIRCLE",
    "GREEN_TRIANGLE",
    "BLUE_CROSS"
]

# Maximum number of patterns that can be stored
MAX_SEQUENCE_LENGTH = 4

# Minimum contour area in pixels
MIN_CONTOUR_AREA = 1500

# Number of consecutive frames required to register a pattern
STABLE_FRAMES = 5

# Number of frames without a pattern before allowing a new one
RELEASE_FRAMES = 8


# HSV color ranges (OpenCV Hue: 0-179)
COLOR_RANGES = {

    "red": [
        ((0, 100, 80), (10, 255, 255)),
        ((170, 100, 80), (179, 255, 255))
    ],

    "yellow": [
        ((18, 100, 100), (38, 255, 255))
    ],

    "green": [
        ((35, 80, 70), (85, 255, 255))
    ],
    "blue": [
        ((90, 80, 50), (130, 255, 255))
    ]
}


# ============================================================
# 2. PATTERN DETECTOR
# ============================================================

class PatternDetector:

    def __init__(self):

        self.kernel = np.ones((5, 5), np.uint8)

    # --------------------------------------------------------
    # Convert the image from BGR to HSV
    # --------------------------------------------------------

    def preprocess(self, frame):

        hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

        return hsv

    # --------------------------------------------------------
    # Create a binary mask for a specific color
    # --------------------------------------------------------

    def get_color_mask(self, hsv, color):

        mask = np.zeros(hsv.shape[:2], dtype=np.uint8)

        # Combine all HSV intervals
        for lower, upper in COLOR_RANGES[color]:

            partial_mask = cv2.inRange(
                hsv,
                np.array(lower),
                np.array(upper)
            )

            mask = cv2.bitwise_or(mask, partial_mask)

        # Remove small noise
        mask = cv2.morphologyEx(
            mask,
            cv2.MORPH_OPEN,
            self.kernel
        )

        # Fill small holes
        mask = cv2.morphologyEx(
            mask,
            cv2.MORPH_CLOSE,
            self.kernel
        )

        return mask

    # --------------------------------------------------------
    # Classify the geometric shape of a contour
    # --------------------------------------------------------

    def classify_shape(self, contour):

        perimeter = cv2.arcLength(contour, True)

        if perimeter == 0:
            return None

        area = cv2.contourArea(contour)

        # Approximate contour with a polygon
        epsilon = 0.02 * perimeter

        approx = cv2.approxPolyDP(
            contour,
            epsilon,
            True
        )

        vertices = len(approx)

        # Calculate circularity
        circularity = (
            4 * np.pi * area / (perimeter ** 2)
        )

        # Calculate solidity
        hull = cv2.convexHull(contour)

        hull_area = cv2.contourArea(hull)

        if hull_area == 0:
            return None

        solidity = area / hull_area

        # Triangle: exactly 3 vertices
        if vertices == 3:
            return "TRIANGLE"

        # Circle: high circularity and several vertices
        if circularity > 0.80 and vertices >= 7:
            return "CIRCLE"

        # Cross: several vertices and a concave contour
        if (
            8 <= vertices <= 16
            and 0.35 <= solidity <= 0.80
        ):
            return "CROSS"

        return None

    # --------------------------------------------------------
    # Detect color + geometric shape
    # --------------------------------------------------------

    def detect_patterns(self, frame):

        hsv = self.preprocess(frame)

        detections = []

        for color in COLOR_RANGES:

            mask = self.get_color_mask(hsv, color)

            contours, _ = cv2.findContours(
                mask,
                cv2.RETR_EXTERNAL,
                cv2.CHAIN_APPROX_SIMPLE
            )

            for contour in contours:

                area = cv2.contourArea(contour)

                if area < MIN_CONTOUR_AREA:
                    continue

                shape = self.classify_shape(contour)

                if shape is None:
                    continue

                # Check that the shape matches the expected color
                valid = False

                if color == "red" and shape == "CROSS":
                    valid = True

                elif color == "yellow" and shape == "CIRCLE":
                    valid = True

                elif color == "green" and shape == "TRIANGLE":
                    valid = True

                elif color == "blue" and shape == "CROSS":
                    valid = True

                if not valid:
                    continue

                # Identify the complete pattern
                pattern = f"{color.upper()}_{shape}"

                x, y, w, h = cv2.boundingRect(contour)

                detections.append({
                    "pattern": pattern,
                    "bbox": (x, y, w, h),
                    "area": area
                })

        # Return the largest valid detection
        if detections:

            detections.sort(
                key=lambda detection: detection["area"],
                reverse=True
            )

            return detections[0]

        return None


# ============================================================
# 3. SECURITY DECODER
# ============================================================

class SecurityDecoder:

    def __init__(self):

        self.password = PASSWORD.copy()

        self.sequence = []

        self.access_granted = False

    # --------------------------------------------------------
    # Process a detected pattern
    # --------------------------------------------------------

    def decode_pattern(self, pattern):

        if self.access_granted:
            return "ACCESS_GRANTED"

        # Check memory capacity
        if len(self.sequence) >= MAX_SEQUENCE_LENGTH:

            self.reset()

            return "ACCESS_DENIED"

        # Determine the next expected pattern
        expected = self.password[len(self.sequence)]

        # Correct pattern
        if pattern == expected:

            self.sequence.append(pattern)

            print("Correct pattern:", pattern)

            # Check whether the complete password is correct
            if self.sequence == self.password:

                self.access_granted = True

                print("ACCESS GRANTED!")

                return "ACCESS_GRANTED"

            return "CORRECT"

        # Incorrect pattern
        else:

            print("Incorrect pattern:", pattern)

            self.reset()

            return "ACCESS_DENIED"

    # --------------------------------------------------------
    # Reset the decoder memory
    # --------------------------------------------------------

    def reset(self):

        self.sequence.clear()

        self.access_granted = False

        print("Sequence reset.")

    # --------------------------------------------------------
    # Return sequence progress
    # --------------------------------------------------------

    def get_progress(self):

        return len(self.sequence), len(self.password)


# ============================================================
# 4. MAIN SECURITY SYSTEM
# ============================================================

class SecuritySystem:

    def __init__(self):

        self.detector = PatternDetector()

        self.decoder = SecurityDecoder()

        self.stable_count = 0

        self.release_count = 0

        self.last_candidate = None

        self.pattern_locked = False

        self.current_status = "WAITING..."

    # --------------------------------------------------------
    # Process each camera frame
    # --------------------------------------------------------

    def process_frame(self, frame):

        detection = self.detector.detect_patterns(frame)

        pattern = None
        box = None

        if detection is not None:

            pattern = detection["pattern"]
            box = detection["bbox"]

        # ----------------------------------------------------
        # Register a pattern only after stable detection
        # ----------------------------------------------------

        if pattern is not None:

            self.release_count = 0

            if not self.pattern_locked:

                if pattern == self.last_candidate:

                    self.stable_count += 1

                else:

                    self.last_candidate = pattern
                    self.stable_count = 1

                # Register the pattern after several stable frames
                if self.stable_count >= STABLE_FRAMES:

                    result = self.decoder.decode_pattern(pattern)

                    self.current_status = result

                    self.pattern_locked = True

                    self.stable_count = 0

        else:

            self.stable_count = 0

            self.last_candidate = None

            # Wait until the previous pattern disappears
            if self.pattern_locked:

                self.release_count += 1

                if self.release_count >= RELEASE_FRAMES:

                    self.pattern_locked = False

                    self.release_count = 0

        # ----------------------------------------------------
        # Draw detected pattern
        # ----------------------------------------------------

        if box is not None:

            x, y, w, h = box

            cv2.rectangle(
                frame,
                (x, y),
                (x + w, y + h),
                (0, 255, 0),
                2
            )

            cv2.putText(
                frame,
                pattern,
                (x, max(y - 10, 20)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (0, 255, 0),
                2
            )

        self.draw_interface(frame, pattern)

        return frame

    # --------------------------------------------------------
    # Draw security system information
    # --------------------------------------------------------

    def draw_interface(self, frame, pattern):

        progress, total = self.decoder.get_progress()

        # Sequence progress
        cv2.putText(
            frame,
            f"SECURITY SYSTEM: {progress}/{total}",
            (20, 35),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (255, 255, 255),
            2
        )

        # Current detected pattern
        detected_text = pattern if pattern else "None"

        cv2.putText(
            frame,
            f"Detected: {detected_text}",
            (20, 70),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.55,
            (255, 255, 255),
            2
        )

        # Access status
        if self.decoder.access_granted:

            status = "ACCESS GRANTED"
            color = (0, 255, 0)

        elif self.current_status == "ACCESS_DENIED":

            status = "ACCESS DENIED"
            color = (0, 0, 255)

        else:

            status = "WAITING..."
            color = (0, 255, 255)

        cv2.putText(
            frame,
            status,
            (20, 115),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.9,
            color,
            2
        )

        # Display stored sequence
        sequence_text = " -> ".join(
            self.decoder.sequence
        )

        cv2.putText(
            frame,
            f"Sequence: {sequence_text}",
            (20, 155),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.45,
            (255, 255, 255),
            1
        )


# ============================================================
# 5. CAMERA EXECUTION
# ============================================================

def main():

    camera = cv2.VideoCapture(CAMERA_INDEX)

    if not camera.isOpened():

        raise RuntimeError("Could not open camera")

    security_system = SecuritySystem()

    print("----------------------------------------")
    print("SECURITY SYSTEM STARTED")
    print("----------------------------------------")

    print("Required sequence:")
    print("1. RED CROSS")
    print("2. YELLOW CIRCLE")
    print("3. GREEN TRIANGLE")
    print("4. BLUE CROSS")


    print("----------------------------------------")
    print("Press R to reset the decoder.")
    print("Press Q to exit.")
    print("----------------------------------------")

    while True:

        ret, frame = camera.read()

        if not ret:

            print("Error capturing image.")
            break

        # Process image
        frame = security_system.process_frame(frame)

        # Display camera window
        cv2.imshow(
            "Security System",
            frame
        )

        key = cv2.waitKey(1) & 0xFF

        # Manual reset
        if key == ord("r"):

            security_system.decoder.reset()

            security_system.current_status = "WAITING..."

            security_system.pattern_locked = False

            security_system.stable_count = 0

            security_system.release_count = 0

            security_system.last_candidate = None

        # Exit program
        elif key == ord("q"):

            break

    camera.release()

    cv2.destroyAllWindows()


# ============================================================
# 6. PROGRAM ENTRY POINT
# ============================================================

if __name__ == "__main__":

    main()


----------------------------------------
SECURITY SYSTEM STARTED
----------------------------------------
Required sequence:
1. RED CROSS
2. YELLOW CIRCLE
3. GREEN TRIANGLE
4. BLUE CROSS
----------------------------------------
Press R to reset the decoder.
Press Q to exit.
----------------------------------------
Correct pattern: RED_CROSS
Incorrect pattern: RED_CROSS
Sequence reset.
Sequence reset.
Correct pattern: RED_CROSS
Incorrect pattern: RED_CROSS
Sequence reset.
Sequence reset.
Incorrect pattern: BLUE_CROSS
Sequence reset.
Incorrect pattern: BLUE_CROSS
Sequence reset.
Correct pattern: RED_CROSS
Correct pattern: YELLOW_CIRCLE
Correct pattern: GREEN_TRIANGLE
Correct pattern: BLUE_CROSS
ACCESS GRANTED!
